# Regresión con PyTorch: Predicción de tarifas de taxi en NYC
Este notebook entrena una red neuronal para predecir el costo de un viaje en taxi usando variables continuas y categóricas.

## 1. Importación de librerías
Se cargan las herramientas necesarias para manipulación de datos, visualización, preprocesamiento y construcción del modelo en PyTorch.

In [ ]:
# Importamos librerías fundamentales para ciencia de datos y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Herramientas de Scikit-learn para dividir datos y escalar/preprocesar variables
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Métricas de regresión para evaluar el desempeño del modelo
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# PyTorch y utilidades para construir la red neuronal y manejar datasets
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## 2. Carga y exploración de datos
Leemos el dataset de viajes de taxi y mostramos las primeras filas para conocer su estructura.

In [ ]:
# Cargamos el CSV con los datos de taxis y vemos una muestra inicial
df = pd.read_csv('C:\\Users\\alvar\\Documents\\pytorch_ejemplos\\DATA\\NYCTaxiFares.csv')
df.head()

## 3. Ingeniería de características
Extraemos nuevas variables a partir de la fecha/hora de recogida y calculamos la distancia geográfica del viaje.

In [ ]:
# Convertimos la columna de fecha y hora al tipo datetime para poder extraer componentes
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

In [ ]:
# Ajustamos la zona horaria restando 4 horas (ej: de UTC a hora del Este de EE.UU.)
df['pickup_datetime'] = df['pickup_datetime'] - pd.Timedelta(hours=4)

In [ ]:
# Extraemos la hora del día como entero (0-23)
df['hour'] = df['pickup_datetime'].dt.hour

In [ ]:
# Creamos una variable categórica 'am' o 'pm' según la hora del día
df['am_pm'] = np.where(df['hour'] < 12, 'am', 'pm')

In [ ]:
# Extraemos el día de la semana abreviado en inglés (Mon, Tue, Wed, etc.)
df['weekday'] = df['pickup_datetime'].dt.strftime('%a')

In [ ]:
# Verificamos el resultado de las nuevas columnas creadas
df.head()

In [ ]:
# Función que calcula la distancia entre dos puntos geográficos usando la fórmula de Haversine
# Devuelve la distancia en kilómetros basada en coordenadas (latitud/longitud)
def haversine_distance(df, lat1, long1, lat2, long2):
    r = 6371  # Radio de la Tierra en kilómetros
    phi1 = np.radians(df[lat1])
    phi2 = np.radians(df[lat2])
    lambda1 = np.radians(df[long1])
    lambda2 = np.radians(df[long2])
    delta_phi = phi2 - phi1
    delta_lambda = lambda2 - lambda1

    # Fórmula del haversine: más estable numéricamente para distancias pequeñas
    hav = (1 - np.cos(delta_phi) + np.cos(phi1) * np.cos(phi2) * (1 - np.cos(delta_lambda))) / 2
    theta = 2 * np.arcsin(np.sqrt(hav))
    d = r * theta
    return d

In [ ]:
# Aplicamos la función para obtener la distancia real del recorrido en kilómetros
df['dist_km'] = haversine_distance(
    df, 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude'
)
df.head()

## 4. Definición de columnas y preprocesamiento
Separamos las columnas categóricas, continuas y el objetivo. Luego codificamos y escalamos para PyTorch.

In [ ]:
# Definimos las columnas categóricas, continuas y la variable objetivo
cat_cols = ['hour', 'am_pm', 'weekday']
cont_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'dist_km']
target_col = 'fare_amount'

In [ ]:
# Función de preprocesamiento:
# - Codifica las variables categóricas con LabelEncoder
# - Escala las variables continuas con StandardScaler (media=0, std=1)
# - Convierte todo a tensores de PyTorch
def process_data(df, cat_cols, cont_cols, target_col):
    encoders = {}
    for col in cat_cols:
        encoders[col] = LabelEncoder()
        df[col] = encoders[col].fit_transform(df[col])

    scaler = StandardScaler()
    df[cont_cols] = scaler.fit_transform(df[cont_cols])

    cats = torch.tensor(df[cat_cols].values, dtype=torch.long)
    conts = torch.tensor(df[cont_cols].values, dtype=torch.float)
    y = torch.tensor(df[target_col].values, dtype=torch.float).reshape(-1, 1)

    return cats, conts, y, encoders, scaler

In [ ]:
# Ejecutamos el preprocesamiento y obtenemos los objetos que permitirán transformar nuevos datos luego
cats, conts, y, encoders, scaler = process_data(df, cat_cols, cont_cols, target_col)

In [ ]:
# Dividimos los datos en entrenamiento (80%) y validación (20%) de forma reproducible
cat_train, cat_val, cont_train, cont_val, y_train, y_val = train_test_split(
    cats, conts, y, test_size=0.2, random_state=42
)

In [ ]:
# Creamos datasets de PyTorch y DataLoaders para iterar por lotes (batches)
train_dataset = TensorDataset(cat_train, cont_train, y_train)
val_dataset = TensorDataset(cat_val, cont_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024)

## 5. Definición del modelo
Construimos la red neuronal con embeddings para variables categóricas y capas densas para la regresión.

In [ ]:
# Calculamos los tamaños de embedding para cada columna categórica
# La regla heurística es: min(50, (número_de_categorías + 1) // 2)
emb_szs = [(df[col].nunique(), min(50, (df[col].nunique() + 1) // 2)) for col in cat_cols]
print('Tamaños de embedding:', emb_szs)

In [ ]:
# Modelo de red neuronal para datos tabulares
# Combina embeddings (categóricas) + normalización batch (continuas) + capas densas
class TabularModel(nn.Module):
    def __init__(self, emb_szs, n_cont, out_sz, layers, p=0.5):
        super().__init__()
        # Capa de embedding para cada variable categórica
        self.embeds = nn.ModuleList([nn.Embedding(ni, nf) for ni, nf in emb_szs])
        self.emb_drop = nn.Dropout(p)
        # Normalización por lotes para las variables continuas
        self.bn_cont = nn.BatchNorm1d(n_cont)

        # Construimos las capas densas (fully connected) dinámicamente
        layerlist = []
        n_emb = sum(nf for _, nf in emb_szs)  # Suma total de dimensiones de embedding
        n_in = n_emb + n_cont  # Tamaño de entrada a la primera capa densa

        for layer_size in layers:
            layerlist.extend([
                nn.Linear(n_in, layer_size),
                nn.ReLU(),  # Activación no lineal
                nn.BatchNorm1d(layer_size),  # Estabiliza y acelera el entrenamiento
                nn.Dropout(p)  # Regularización para evitar sobreajuste
            ])
            n_in = layer_size

        # Capa de salida: sin activación (regresión de un valor continuo)
        layerlist.append(nn.Linear(layers[-1], out_sz))
        self.layers = nn.Sequential(*layerlist)

    def forward(self, x_cat, x_cont):
        # Obtenemos el embedding de cada columna categórica y los concatenamos
        embeddings = [e(x_cat[:, i]) for i, e in enumerate(self.embeds)]
        x = torch.cat(embeddings, 1)
        x = self.emb_drop(x)
        # Normalizamos las continuas y concatenamos todo
        x_cont = self.bn_cont(x_cont)
        return self.layers(torch.cat([x, x_cont], 1))

## 6. Entrenamiento del modelo
Configuramos el dispositivo (GPU si está disponible), la función de pérdida, el optimizador y el scheduler. Luego entrenamos y registramos métricas.

In [ ]:
# Detectamos si hay GPU disponible; de lo contrario usamos CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo usado: {device}')

In [ ]:
# Instanciamos el modelo y lo movemos al dispositivo (CPU/GPU)
model = TabularModel(emb_szs, len(cont_cols), 1, [128, 64, 32], p=0.2).to(device)

# Pérdida: MSE para regresión
criterion = nn.MSELoss()
# Optimizador Adam con tasa de aprendizaje inicial
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Reduce la tasa de aprendizaje si la pérdida de validación no mejora en 10 épocas
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

In [ ]:
# Listas para graficar la evolución del entrenamiento
train_losses = []
val_losses = []
epochs = 30

for epoch in range(epochs):
    # --- Fase de entrenamiento ---
    model.train()
    train_loss = 0.0
    for cat_batch, cont_batch, y_batch in train_loader:
        cat_batch = cat_batch.to(device)
        cont_batch = cont_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(cat_batch, cont_batch)
        loss = torch.sqrt(criterion(outputs, y_batch))  # RMSE por batch
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * cat_batch.size(0)  # Acumulamos ponderado por batch

    # --- Fase de validación ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for cat_batch, cont_batch, y_batch in val_loader:
            cat_batch = cat_batch.to(device)
            cont_batch = cont_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(cat_batch, cont_batch)
            loss = torch.sqrt(criterion(outputs, y_batch))
            val_loss += loss.item() * cat_batch.size(0)

    # Promediamos por el número total de muestras
    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Ajustamos la tasa de aprendizaje según la pérdida de validación
    scheduler.step(val_loss)

    print(f'Época {epoch+1:02d}/{epochs} | Train RMSE: {train_loss:.4f} | Val RMSE: {val_loss:.4f}')

## 7. Visualización del entrenamiento
Graficamos cómo evolucionaron las pérdidas de entrenamiento y validación a lo largo de las épocas.

In [ ]:
# Graficamos RMSE de entrenamiento vs validación por época
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_losses, label='Train RMSE', marker='o')
plt.plot(range(1, epochs + 1), val_losses, label='Val RMSE', marker='s')
plt.xlabel('Época')
plt.ylabel('RMSE')
plt.title('Evolución del entrenamiento')
plt.legend()
plt.grid(True)
plt.show()

## 8. Métricas de regresión
Calculamos MSE, RMSE, MAE y R² sobre el conjunto de validación para evaluar el modelo cuantitativamente.

In [ ]:
# Obtenemos predicciones sobre todo el conjunto de validación
model.eval()
with torch.no_grad():
    # Movemos los datos de validación al mismo dispositivo que el modelo
    cat_val_dev = cat_val.to(device)
    cont_val_dev = cont_val.to(device)
    y_val_dev = y_val.to(device)

    y_pred = model(cat_val_dev, cont_val_dev)

# Convertimos tensores a arrays de NumPy para usar sklearn.metrics
y_true_np = y_val_dev.cpu().numpy().flatten()
y_pred_np = y_pred.cpu().numpy().flatten()

# Calculamos métricas estándar de regresión
mse = mean_squared_error(y_true_np, y_pred_np)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true_np, y_pred_np)
r2 = r2_score(y_true_np, y_pred_np)

print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

## 9. Guardar y cargar el modelo
Guardamos los pesos, la arquitectura y los objetos de preprocesamiento para poder reutilizarlos sin reentrenar.

In [ ]:
import os
import json

# Creamos una carpeta para almacenar el modelo y sus artefactos
save_dir = 'modelo_taxi'
os.makedirs(save_dir, exist_ok=True)

# Guardamos el estado del modelo (pesos)
torch.save(model.state_dict(), os.path.join(save_dir, 'model_weights.pth'))

# Guardamos información necesaria para reconstruir el modelo y preprocesar nuevos datos
metadata = {
    'emb_szs': emb_szs,
    'n_cont': len(cont_cols),
    'out_sz': 1,
    'layers': [128, 64, 32],
    'p': 0.2,
    'cat_cols': cat_cols,
    'cont_cols': cont_cols,
    'target_col': target_col,
}
with open(os.path.join(save_dir, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

# Guardamos los encoders y el scaler con joblib (estándar en sklearn)
import joblib
joblib.dump(encoders, os.path.join(save_dir, 'encoders.joblib'))
joblib.dump(scaler, os.path.join(save_dir, 'scaler.joblib'))

print(f'Modelo y artefactos guardados en: {os.path.abspath(save_dir)}')

In [ ]:
# --- Código para cargar el modelo guardado ---
import json
import joblib

load_dir = 'modelo_taxi'

# Leemos la metadata del modelo
with open(os.path.join(load_dir, 'model_metadata.json'), 'r') as f:
    metadata = json.load(f)

# Recreamos la arquitectura con los mismos parámetros
loaded_model = TabularModel(
    emb_szs=metadata['emb_szs'],
    n_cont=metadata['n_cont'],
    out_sz=metadata['out_sz'],
    layers=metadata['layers'],
    p=metadata['p']
).to(device)

# Cargamos los pesos entrenados
loaded_model.load_state_dict(torch.load(os.path.join(load_dir, 'model_weights.pth'), map_location=device))
loaded_model.eval()

# Cargamos los objetos de preprocesamiento
loaded_encoders = joblib.load(os.path.join(load_dir, 'encoders.joblib'))
loaded_scaler = joblib.load(os.path.join(load_dir, 'scaler.joblib'))

print('Modelo y preprocesadores cargados correctamente.')

## 10. Inferencia sobre nuevos datos
Creamos una clase `TripPredictor` que encapsula todo el preprocesamiento y predicción para estimar el costo de un nuevo viaje.

In [ ]:
class TripPredictor:
    """
    Clase para predecir tarifas de taxi a partir de datos crudos de un viaje.
    Realiza automáticamente la ingeniería de características y el preprocesamiento
    antes de pasar los datos al modelo entrenado.
    """

    def __init__(self, model, encoders, scaler, device, cont_cols, cat_cols):
        self.model = model
        self.model.eval()
        self.encoders = encoders
        self.scaler = scaler
        self.device = device
        self.cont_cols = cont_cols
        self.cat_cols = cat_cols

    @staticmethod
    def _haversine(lat1, lon1, lat2, lon2):
        """Calcula distancia en km entre dos coordenadas."""
        r = 6371
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
        c = 2 * np.arcsin(np.sqrt(a))
        return r * c

    def preprocess(self, pickup_datetime, pickup_longitude, pickup_latitude,
                   dropoff_longitude, dropoff_latitude, passenger_count):
        """
        A partir de los datos crudos del viaje, genera las mismas características
        que se usaron durante el entrenamiento (hour, am_pm, weekday, dist_km).
        Retorna un diccionario con los valores originales y procesados.
        """
        # Convertimos la fecha/hora y ajustamos zona horaria (mismo ajuste del entrenamiento)
        dt = pd.to_datetime(pickup_datetime) - pd.Timedelta(hours=4)

        # Extraemos las características temporales
        hour = dt.hour
        am_pm = 'am' if hour < 12 else 'pm'
        weekday = dt.strftime('%a')

        # Calculamos la distancia del trayecto con Haversine
        dist_km = self._haversine(
            pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude
        )

        # Armamos un DataFrame con las columnas en el mismo orden del entrenamiento
        data = {
            'hour': [hour],
            'am_pm': [am_pm],
            'weekday': [weekday],
            'pickup_longitude': [pickup_longitude],
            'pickup_latitude': [pickup_latitude],
            'dropoff_longitude': [dropoff_longitude],
            'dropoff_latitude': [dropoff_latitude],
            'passenger_count': [passenger_count],
            'dist_km': [dist_km],
        }
        df_new = pd.DataFrame(data)

        # Aplicamos LabelEncoder usando los encoders entrenados
        for col in self.cat_cols:
            le = self.encoders[col]
            # Si aparece una categoría desconocida, la marcamos con la primera clase como fallback
            df_new[col] = df_new[col].apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else 0
            )

        # Escalamos las variables continuas con el scaler entrenado
        df_new[self.cont_cols] = self.scaler.transform(df_new[self.cont_cols])

        # Convertimos a tensores de PyTorch
        cat_tensor = torch.tensor(df_new[self.cat_cols].values, dtype=torch.long).to(self.device)
        cont_tensor = torch.tensor(df_new[self.cont_cols].values, dtype=torch.float).to(self.device)

        return {
            'cat_tensor': cat_tensor,
            'cont_tensor': cont_tensor,
            'hour': hour,
            'am_pm': am_pm,
            'weekday': weekday,
            'dist_km': float(dist_km),
            'scaled_df': df_new
        }

    def predict(self, pickup_datetime, pickup_longitude, pickup_latitude,
                dropoff_longitude, dropoff_latitude, passenger_count):
        """
        Ejecuta el preprocesamiento completo y devuelve la predicción de tarifa,
        junto con los detalles de las características procesadas.
        """
        info = self.preprocess(
            pickup_datetime, pickup_longitude, pickup_latitude,
            dropoff_longitude, dropoff_latitude, passenger_count
        )

        with torch.no_grad():
            pred = self.model(info['cat_tensor'], info['cont_tensor'])

        fare = float(pred.cpu().item())
        return {
            'fare_predicted': fare,
            'hour': info['hour'],
            'am_pm': info['am_pm'],
            'weekday': info['weekday'],
            'dist_km': info['dist_km'],
            'scaled_features': info['scaled_df']
        }

In [ ]:
# Instanciamos el predictor con el modelo entrenado y los preprocesadores
predictor = TripPredictor(
    model=model,
    encoders=encoders,
    scaler=scaler,
    device=device,
    cont_cols=cont_cols,
    cat_cols=cat_cols
)

# Ejemplo: predecir el costo de un viaje nuevo
resultado = predictor.predict(
    pickup_datetime='2010-04-19 08:17:56',
    pickup_longitude=-73.992365,
    pickup_latitude=40.730521,
    dropoff_longitude=-73.975499,
    dropoff_latitude=40.744746,
    passenger_count=1
)

print(f"Tarifa estimada: ${resultado['fare_predicted']:.2f}")
print(f"Hora extraída: {resultado['hour']} ({resultado['am_pm']})")
print(f"Día de la semana: {resultado['weekday']}")
print(f"Distancia calculada: {resultado['dist_km']:.2f} km")